# Scamless - Training Run (GPU)

Runtime > Change runtime type > **T4 GPU**, then Run all.
Clones the repo, builds the dataset, fine-tunes, exports int8 ONNX, evals, gates, saves artifacts to Google Drive.

In [ ]:
%cd /content
!git clone https://github.com/RishavGupta01/Scamless.git scamless
%cd /content/scamless

In [ ]:
!pip install -q -e "training[dev]"

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
import pathlib

drive.mount("/content/drive")
DRIVE_ART = pathlib.Path("/content/drive/MyDrive/scamless/artifacts")
DRIVE_ART.mkdir(parents=True, exist_ok=True)
print("artifacts will persist to", DRIVE_ART)

In [ ]:
!python -m scamless.data.fetch_all
!python -m scamless.data.build

In [ ]:
import pathlib
import pandas as pd

from scamless.model.config import TrainConfig
from scamless.model.train import train_model

processed = pathlib.Path("training/data/processed")
cfg = TrainConfig(output_dir="artifacts/model_v1")
train_df = pd.read_parquet(processed / "messages_train.parquet")
val_df = pd.read_parquet(processed / "messages_val.parquet")

model, tokenizer, metrics = train_model(train_df, val_df, cfg)
model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(metrics)

In [ ]:
!python -m scamless.model.export_onnx --model-dir artifacts/model_v1

In [ ]:
import json

import pandas as pd

from scamless.eval.baseline import heuristic_predict
from scamless.eval.harness import compute_metrics

# trained model metrics + release gate (gate exits nonzero on failure)
!python -m scamless.eval.run_eval --mode onnx --model-dir artifacts/model_v1
!python -m scamless.eval.gate --metrics artifacts/eval/metrics.json

# heuristic baseline for comparison (does NOT overwrite metrics.json)
test_df = pd.read_parquet("training/data/processed/messages_test.parquet")
base = compute_metrics(test_df, [heuristic_predict(str(t)) for t in test_df["text"]])
print("baseline macro_f1:", base["macro_f1"], "fp_rate:", base["false_positive_rate"])
print("trained:", json.loads(open("artifacts/eval/metrics.json").read()))

In [ ]:
import shutil

shutil.copytree("artifacts", DRIVE_ART / "model_v1_run", dirs_exist_ok=True)
print("saved to", DRIVE_ART / "model_v1_run")